<a href="https://colab.research.google.com/github/kshitij-nandanwar/GenAI/blob/main/Prac_3_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LSTM NEXT WORD PREDICTION - GOOGLE COLAB
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


# ============================================================
# 2. CHECK TENSORFLOW VERSION
# ============================================================

print("TensorFlow version:", tf.__version__)


# ============================================================
# 3. LOAD TEXT DATA
# ============================================================

file_path = "/content/Untitled.txt"

print("\nAttempting to load text from:")
print(file_path)

if not os.path.exists(file_path):
    print("\nERROR: File not found!")
    print("\nFiles available in /content:")
    print(os.listdir("/content"))
    raise FileNotFoundError(
        f"LSTM.txt was not found at {file_path}"
    )

# Read the text file
with open(
    file_path,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    raw_text = f.read()

print("\nFile loaded successfully!")
print("Total characters:", len(raw_text))

print("\nFirst 500 characters:")
print("-" * 60)
print(raw_text[:500])
print("-" * 60)


# ============================================================
# 4. BASIC TEXT PREPROCESSING
# ============================================================

text = raw_text.lower()

# Remove unnecessary whitespace
text = " ".join(text.split())

print("\nPreprocessed text length:", len(text))


# ============================================================
# 5. TOKENIZATION
# ============================================================

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index

vocab_size = len(word_index) + 1

print("\nVocabulary Size:", vocab_size)


# ============================================================
# 6. CREATE N-GRAM SEQUENCES
# ============================================================

input_sequences = []

# Split text into sentences
sentences = text.split(".")

print("\nNumber of sentences:", len(sentences))

for sentence in sentences:
    sentence = sentence.strip()

    if not sentence:
        continue

    # Convert sentence into integer tokens
    token_list = tokenizer.texts_to_sequences(
        [sentence]
    )[0]

    # Create n-gram sequences
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(
            n_gram_sequence
        )

print(
    "Total input sequences:",
    len(input_sequences)
)


# ============================================================
# 7. CHECK SEQUENCES
# ============================================================

if len(input_sequences) == 0:
    raise ValueError(
        "No input sequences were created. "
        "Please check the contents of LSTM.txt."
    )


# ============================================================
# 8. FIND MAXIMUM SEQUENCE LENGTH
# ============================================================

max_sequence_len = max(
    len(sequence)
    for sequence in input_sequences
)

print(
    "Maximum sequence length:",
    max_sequence_len
)


# ============================================================
# 9. PAD SEQUENCES
# ============================================================

padded_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding="pre"
    )
)

print(
    "\nPadded sequences shape:",
    padded_sequences.shape
)


# ============================================================
# 10. CREATE X AND Y
# ============================================================

# X = input words
# y = next word

X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 11. ONE-HOT ENCODE Y
# ============================================================

y = to_categorical(
    y,
    num_classes=vocab_size
)

print(
    "One-hot encoded y shape:",
    y.shape
)


# ============================================================
# 12. BUILD LSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    )
)

model.add(
    LSTM(
        256
    )
)

model.add(
    Dense(
        vocab_size,
        activation="softmax"
    )
)


# ============================================================
# 13. COMPILE MODEL
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


# ============================================================
# 14. MODEL SUMMARY
# ============================================================

print("\nModel Summary:")
print("=" * 60)

model.summary()


# ============================================================
# 15. TRAIN MODEL
# ============================================================

print("\n")
print("=" * 60)
print("STARTING MODEL TRAINING")
print("=" * 60)

history = model.fit(
    X,
    y,
    epochs=50,
    batch_size=64,
    verbose=1
)

print("\n")
print("=" * 60)
print("MODEL TRAINING COMPLETE")
print("=" * 60)


# ============================================================
# 16. CREATE REVERSE WORD INDEX
# ============================================================

reverse_word_index = {
    index: word
    for word, index in tokenizer.word_index.items()
}


# ============================================================
# 17. NEXT WORD PREDICTION FUNCTION
# ============================================================

def predict_next_word(seed_text, n_words):
    predicted_sentence = seed_text.lower()

    for _ in range(n_words):
        # Convert text to integer tokens
        token_list = tokenizer.texts_to_sequences(
            [predicted_sentence]
        )[0]

        # Keep only the latest tokens if too long
        if len(token_list) > max_sequence_len - 1:
            token_list = token_list[
                -(max_sequence_len - 1):
            ]

        # Pad sequence
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding="pre"
        )

        # Predict probabilities
        predicted_probs = model.predict(
            token_list,
            verbose=0
        )[0]

        # Select word with highest probability
        predicted_word_index = np.argmax(
            predicted_probs
        )

        # Convert index back to word
        output_word = reverse_word_index.get(
            predicted_word_index,
            ""
        )

        if output_word:
            predicted_sentence += (
                " " + output_word
            )
        else:
            break

    return predicted_sentence


print(
    "\nPrediction function created successfully."
)


# ============================================================
# 18. USER INPUT
# ============================================================

print("\n")
print("=" * 60)
print("NEXT WORD PREDICTION")
print("=" * 60)

user_seed_text = input(
    "Enter your seed text: "
).lower().strip()

try:
    num_words_to_predict = int(
        input("How many words to predict? ")
    )

    if num_words_to_predict <= 0:
        print(
            "\nPlease enter a number greater than 0."
        )
    else:
        predicted_sentence = predict_next_word(
            user_seed_text,
            num_words_to_predict
        )

        print("\n")
        print("=" * 60)
        print("PREDICTION RESULT")
        print("=" * 60)

        print("\nSeed Text:")
        print(user_seed_text)

        print("\nPredicted Sentence:")
        print(predicted_sentence)
        print("\n")

except ValueError:
    print(
        "\nInvalid input. "
        "Please enter an integer."
    )


# ============================================================
# 19. SAVE TRAINED MODEL
# ============================================================

model_path = "/content/lstm_next_word_model.keras"

model.save(model_path)

print(
    "\nModel saved successfully at:"
)
print(model_path)

TensorFlow version: 2.20.0

Attempting to load text from:
/content/Untitled.txt

File loaded successfully!
Total characters: 581886

First 500 characters:
------------------------------------------------------------

Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charact
------------------------------------------------------------

Preprocessed text length: 578750

Vocabulary Size: 8932

Number of sentences: 6434
Total input sequences: 104819
Maximum sequence length: 105

Padded sequences shape: (104819, 105)

X shape: (104819, 104)
y shape: (104819

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)



STARTING MODEL TRAINING
